# 1. Setup
Compositional prototype retrieval with **DINOv2 ViT-B/14**. Frozen backbone; learned crop/disease projections; partner-balanced prototypes from training only. See `docs/approach4_scope.md` for methodology, task distribution and limitations. Run from the repository root. In Colab use `colab/README.md`.


In [ ]:
import os
import sys
from pathlib import Path
import torch
root = Path.cwd().parent if Path.cwd().name == "approaches" else Path.cwd()
sys.path.insert(0, str(root))
from common import config
from common.config import set_seed, NUM_CROPS, NUM_DISEASES
from common.dataset import get_loaders, get_pairs
from common.evaluate import evaluate
from approaches.prototype_model import CompositionalPrototypeRetrieval, retrieve
from approaches.prototype_training import train_model, save_checkpoint
MODEL_NAME = "approach4_compositional_prototype_retrieval"
DEBUG = os.environ.get("CPR_DEBUG", "1") == "1"
HP = {"backbone": "vit_base_patch14_dinov2.lvd142m", "image_size": 224,
      "embedding_dim": 256, "lr": 1e-3, "epochs": 30, "batch_size": 32,
      "head_batch_size": 256, "optimizer": "AdamW", "weight_decay": 1e-4,
      "temperature": 0.1, "crop_weight": 0.5, "prototype_loss_weight": 1.0,
      "seen_bias": 0.0, "num_seen_pairs": 323, "seed": 42,
      "frozen_backbone": True, "cached_augmented_views": 1, "debug": DEBUG}
set_seed(HP["seed"])
RUN_NAME = MODEL_NAME + ("_debug" if DEBUG else "")
print({"run": RUN_NAME, "device": str(config.DEVICE), "hp": HP})
if os.environ.get("CPR_OUTPUT_DIR"):
    config.RESULTS_DIR = Path(os.environ["CPR_OUTPUT_DIR"]) / "results"
    config.CHECKPOINT_DIR = Path(os.environ["CPR_OUTPUT_DIR"]) / "checkpoints"


# 2. Data
Use the team loader, normalization, augmentation, seed and validation split unchanged. Reference prototypes use deterministic transforms applied **only to the training subset**. Validation and test images never enter the memory.


In [ ]:
train_loader, val_loader, seen_loader, unseen_loader = get_loaders(
    HP["batch_size"], norm="imagenet", debug=DEBUG)
pairs = get_pairs()
assert pairs.shape == (333, 2)
print({name: len(loader.dataset) for name, loader in zip(
    ["train", "val", "seen", "unseen"], [train_loader, val_loader, seen_loader, unseen_loader])})


# 3. Model
For pair `(c,d)`, score = `w*cos(z_crop,p_crop[c])/T + (1-w)*cos(z_disease,p_disease[d])/T - seen_bias*is_seen`. There is no learned pair classifier: the same composition rule scores seen and unseen pairs. Missing concepts in DEBUG are masked; full training fails clearly if any candidate concept has no support.


In [ ]:
model = CompositionalPrototypeRetrieval(HP, pairs, NUM_CROPS, NUM_DISEASES).to(config.DEVICE)
@torch.no_grad()
def predict(images):
    model.eval()
    return model(images.to(next(model.parameters()).device)).float().cpu()


# 4. Training
Cache one augmented DINO view per training image, plus deterministic support and validation features. Train the two projection heads with auxiliary concept cross-entropy and prototype cross-entropy. Refresh partner-balanced support means after each epoch. Keep the best validation Top-1 across all 333 pairs. The backbone remains frozen.


In [ ]:
history, memory = train_model(model, train_loader, val_loader, HP, debug=DEBUG)
save_checkpoint(model, memory, history, HP, RUN_NAME, DEBUG)
print("Best validation Top-1:", max(row["val_top1"] for row in history))


# 5. Evaluation
Run shared Top-1, Top-5, harmonic mean, seen confusion matrix and robustness evaluation after model selection. DEBUG results are a smoke test, not reportable benchmark results. Shared robustness currently covers the seen split; synthetic corruptions do not establish real-field generalization.


In [ ]:
results = evaluate(predict, seen_loader, unseen_loader, RUN_NAME, HP, robust=True)
images = next(iter(seen_loader))[0][:1]
examples = retrieve(model, images, memory, k=5)
print("Supporting training images:", examples)


# 6. Save
Checkpoint includes weights, prototypes, pair ordering, HP, run history, environment and training-only retrieval memory. Load via `load_checkpoint(path)`; no pretrained download is required on restore. Outputs and data stay outside Git.


In [ ]:
save_checkpoint(model, memory, history, HP, RUN_NAME, DEBUG)
print("Checkpoint:", config.CHECKPOINT_DIR / f"{RUN_NAME}.pt")
print("Results:", config.RESULTS_DIR / f"{RUN_NAME}.json")
